# Thin MCP server demo

Anton Antonov   
May 2026

---

## Introduction

This notebook has a complete usage example of a _thin_ Model Context Protocol (MCP) client of the Raku package "MCP::Client".

The MCP server is created and run in Python -- see the file ["simple_mcp_server.py"](../resources/simple_mcp_server.py).
That MCP server provides the methods `add`, `random_words_list`, `random_pet_names_list`, and `random_pretentious_job_title_list`.
The corresponding Python packages have to be installed -- see the `from ... import ...` statements in that Python file.

"MCP::Client" provides the class `MCP::Client` which creates from MCP server's methods `LLM::Tool` objects that can be used with Raku's Large Language Model (LLM) framework implemented with ["LLM::Functions"](https://raku.land/zef:antononcube/LLM::Functions), ["LLM::Prompts"](https://raku.land/zef:antononcube/LLM::Prompts), ["Text::SubParsers"](https://raku.land/zef:antononcube/Text::SubParsers); see [AA1÷4, AAp1÷3].

**Remark:** For all of the code in "one file" see the Raku file ["MCP-client-demo.raku"](https://github.com/antononcube/Raku-MCP-Client/blob/main/examples/MCP-client-demo.raku).

---

## Setup

Load the packages used below.

In [ ]:
use LLM::Functions;
use LLM::Prompts;
use Text::SubParsers;
use Data::Translators;

use MCP::Client;

---

## Example

### Setup and initialization

Setup MCP server process creation command elements:

In [28]:
my $python = $*HOME ~ '/miniforge3/envs/ADK13/bin/python';
my $mcp-server-file =  $*HOME ~ '/Python/MCP/simple_mcp_server.py';

die "Cannot find Python executable ⎡$python⎦." unless $python.IO.f;
die "Cannot find MCP server file ⎡$mcp-server-file⎦." unless $mcp-server-file.IO.f;

# say "Using the Python executable ⎡$python⎦.";
# say "Using the MCP server file ⎡$mcp-server-file⎦.";

()

Create the MCP client object:

In [29]:
my Bool:D $echo = False;
my Numeric:D $sleep = 0.7;
my $client = MCP::Client.new(:$echo, :$sleep);

sink $client.start([$python, '-i', $mcp-server-file]);

Initialize the client:

In [30]:
$client.initialize;

True

### Tools discovery and creation

Get the MCP server tools list (and tabulate parts of it):

In [34]:
#% html
my @mcp-tools = |$client.list-tools();
@mcp-tools
==> to-html(field-names => <name description>, align => 'left')

name,description
add,Add two numbers. Args: a: First number. b: Second number.
random_words_list,"Generate a list of random words. Args: n: number of random words do generate. kind: kind of word, one of ""Any"", ""Common"", ""Known"", ""Stopword""."
random_pet_names_list,"Generate a list of random pet names. Args: n: number of random pet names do generate. species: species, one of ""Cat"", ""Dog"", ""Goat"", ""Pig"", or Non."
random_pretentious_job_title_list,"Generate a list of random pretentious job titles. Args: n: number of random pretentious job titles. lang: language, one of ""Bulgarian"", ""English"", or None."


Make `LLM::Tool` objects:

In [35]:
my @tools = @mcp-tools.map({ $client.to-llm-tool($_) });

[LLMTool(add, 
Add two numbers.

Args:
    a: First number.
    b: Second number.
) LLMTool(random_words_list, 
Generate a list of random words.

Args:
    n: number of random words do generate.
    kind: kind of word, one of "Any", "Common", "Known", "Stopword".
) LLMTool(random_pet_names_list, 
Generate a list of random pet names.

Args:
    n: number of random pet names do generate.
    species: species, one of "Cat", "Dog", "Goat", "Pig", or Non.
) LLMTool(random_pretentious_job_title_list, 
Generate a list of random pretentious job titles.

Args:
    n: number of random pretentious job titles.
    lang: language, one of "Bulgarian", "English", or None.
)]

### LLM invocations

Using an LLM request generate a list of random words (via an MCP server provided tool):

In [36]:
my $conf = llm-configuration('ChatGPT', model => 'gpt-4.1-mini', :@tools);
say llm-synthesize('Generate a list of 12 random cat pet names.', e => $conf);

Here is a list of 12 random cat pet names:
1. Tiggie
2. Zoey
3. Duka
4. Sasha
5. Artemis
6. Charlie
7. Comet
8. Neko
9. Soumaya
10. Gnomi
11. Moby Dave
12. Jack


Generate a JSON object of random pretentious jobs, and deserialize and print it:

In [ ]:
#% html
my $res = llm-synthesize([
    'Generate a list of six random bullshit jobs, in English, and three in Bulgarian.)',
    llm-prompt('NothingElse')('JSON')
    ],
    e => $conf,
    form => sub-parser('JSON'):drop
);

$res
==> to-html(align => 'left')

Bulgarian,Директен Представител на МрежиСтарши Администратор по ИзследванияКорпоративен Посредник по Идентичност
English,National Solutions ManagerDirect Configuration OfficerProduct Applications ArchitectInvestor Directives AssistantHuman Brand AdministratorRegional Research Coordinator


### Stopping the MCP server process

Kill the MCP client process:

In [45]:
$client.kill;

1

----

## References


## Articles, blog posts

[AA1] Anton Antonov,
["LLM function calling workflows (Part 1, OpenAI)"](https://rakuforprediction.wordpress.com/2025/06/01/llm-function-calling-workflows-part-1-openai/),
(2025),
[RakuForPrediction at WordPress](https://rakuforprediction.wordpress.com).

[AA2] Anton Antonov,
["LLM function calling workflows (Part 2, Google's Gemini)"](https://rakuforprediction.wordpress.com/2025/06/01/llm-function-calling-workflows-part-2-google-gemini/),
(2025),
[RakuForPrediction at WordPress](https://rakuforprediction.wordpress.com).

[AA3] Anton Antonov,
["LLM function calling workflows (Part 3, Facilitation)"](https://rakuforprediction.wordpress.com/2025/06/08/llm-function-calling-workflows-part-3-facilitation/),
(2025),
[RakuForPrediction at WordPress](https://rakuforprediction.wordpress.com).

[AA4] Anton Antonov,
["LLM function calling workflows (Part 4, Universal specs)"](https://rakuforprediction.wordpress.com/2025/09/28/llm-function-calling-workflows-part-4-universal-specs/),
(2025),
[RakuForPrediction at WordPress](https://rakuforprediction.wordpress.com).

### Packages

[AAp1] Anton Antonov, [LLM::Functions, Raku package](https://github.com/antononcube/Raku-LLM-Functions), (2023-2026), [GitHub/antononcube](https://github.com/antononcube).

[AAp2] Anton Antonov, [LLM::Prompts, Raku package](https://github.com/antononcube/Raku-LLM-Prompts), (2023-2026), [GitHub/antononcube](https://github.com/antononcube).

[AAp3] Anton Antonov, [Text::SubParsers, Raku package](https://github.com/antononcube/Raku-Text-SubParsers), (2023), [GitHub/antononcube](https://github.com/antononcube).